# Làm sạch dữ liệu (Data Cleaning)

**Đề tài:** Dự báo doanh thu và giá vốn hàng bán theo ngày
**Bước trước:** [Xây dựng Từ điển dữ liệu và Sơ đồ ERD](Dictionary-va-ERD.ipynb)

---

## Nguyên tắc

> **Mọi thao tác làm sạch đều phải có căn cứ từ bước Từ điển dữ liệu.**
> Không xóa cột nào vì cảm tính — mỗi lần xóa đều dẫn ra được con số đã kiểm chứng.

Bước Từ điển đã tìm ra **17 cảnh báo chất lượng dữ liệu**. Notebook này xử lý từng cảnh báo đó.

## Ba ràng buộc tự đặt ra

| Ràng buộc | Lý do |
|---|---|
| **Không xóa dòng nào** | Dữ liệu bẩn ở đây là *cột thừa*, không phải *dòng lỗi* |
| **Không sửa dữ liệu gốc** | Ghi ra thư mục `data_clean/`, giữ nguyên `data/` |
| **Kiểm chứng lại sau khi làm sạch** | Toàn vẹn tham chiếu và biến mục tiêu phải không đổi |

---

## 0. Nạp dữ liệu

In [1]:
import pandas as pd, numpy as np, warnings, os
warnings.filterwarnings('ignore')
pd.set_option('display.width', 120)

VAO, RA = 'data/', 'data_clean/'
os.makedirs(RA, exist_ok=True)

TEN = ['orders','order_items','payments','shipments','returns','reviews',
       'customers','products','geography','promotions','inventory',
       'sales','web_traffic','sample_submission']

b = {t: pd.read_csv(VAO + t + '.csv', low_memory=False) for t in TEN}
truoc = {t: b[t].shape for t in TEN}
nhat_ky = []

def ghi(bang, thao_tac, chi_tiet, can_cu):
    nhat_ky.append({'Bang': bang, 'Thao tac': thao_tac,
                    'Chi tiet': chi_tiet, 'Can cu': can_cu})

print('Da nap', len(b), 'bang |', format(sum(v[0] for v in truoc.values()), ',d'),
      'dong |', sum(v[1] for v in truoc.values()), 'cot')

Da nap 14 bang | 2,960,736 dong | 96 cot


---

## Bước 1 — Ép kiểu ngày

**Vấn đề:** CSV lưu ngày dưới dạng chuỗi. Nếu không ép kiểu thì không so sánh hay tính hiệu ngày được.

In [2]:
COT_NGAY = {
    'orders': ['order_date'], 'customers': ['signup_date'],
    'shipments': ['ship_date','delivery_date'], 'returns': ['return_date'],
    'reviews': ['review_date'], 'inventory': ['snapshot_date'],
    'promotions': ['start_date','end_date'],
    'sales': ['Date'], 'web_traffic': ['date'], 'sample_submission': ['Date'],
}
n = 0
for t, cols in COT_NGAY.items():
    for c in cols:
        b[t][c] = pd.to_datetime(b[t][c]); n += 1

ghi('(nhieu bang)', 'Ep kieu ngay', str(n) + ' cot chuoi -> datetime',
    'CSV luu ngay duoi dang chuoi')
print('Da ep kieu', n, 'cot ngay')

Da ep kieu 12 cot ngay


---

## Bước 2 — Thêm khóa thay thế cho `order_items` ⭐

**Căn cứ:** bước Từ điển phát hiện **16 cặp `(order_id, product_id)` bị trùng** trên 714.669 dòng,
nên bảng này **không có khóa tự nhiên hợp lệ**.

**Hậu quả nếu bỏ qua:** nối bảng bằng cặp đó sẽ làm **nhân dòng**, doanh thu bị tính lố mà không báo lỗi.

**Cách xử lý:** thêm khóa thay thế `order_item_id` chạy từ 1 đến hết.

In [3]:
oi = b['order_items']
n_trung = len(oi) - len(oi.drop_duplicates(['order_id','product_id']))
print('So cap (order_id, product_id) bi trung:', n_trung)

oi.insert(0, 'order_item_id', np.arange(1, len(oi) + 1))
print('Da them khoa thay the order_item_id — duy nhat:', oi.order_item_id.is_unique)

ghi('order_items', 'Them khoa thay the', 'order_item_id (1..' + str(len(oi)) + ')',
    str(n_trung) + ' cap bi trung -> khong co khoa hop le')
oi.head(3)

So cap (order_id, product_id) bi trung: 16
Da them khoa thay the order_item_id — duy nhat: True


,order_item_id,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,1,2400,7,1138.22,0.0,NaN,NaN
1,2,2,609,7,10166.25,0.0,NaN,NaN
2,3,3,396,3,11220.33,0.0,NaN,NaN


---

## Bước 3 — Loại cột hằng số

**Vấn đề:** cột chỉ có một giá trị duy nhất thì **phương sai bằng 0**, không mang thông tin gì.

Thay vì liệt kê tay, em quét toàn bộ 96 cột để tìm.

In [4]:
hang_so = [(t, c) for t in TEN for c in b[t].columns
           if b[t][c].nunique(dropna=False) == 1]

for t, c in hang_so:
    gt = b[t][c].iloc[0]
    print('  ' + t + '.' + c, '-> luon bang', gt)
    b[t] = b[t].drop(columns=[c])
    ghi(t, 'Loai cot hang so', c + ' (luon = ' + str(gt) + ')',
        'Phuong sai = 0, khong mang thong tin')

print()
print('Da loai', len(hang_so), 'cot hang so')

  inventory.reorder_flag -> luon bang 0

Da loai 1 cot hang so


---

## Bước 4 — Loại cột dư thừa

**Cột dư thừa** là bản sao nguyên văn của cột ở bảng khác. Bước Từ điển đã kiểm chứng **trùng khớp 100%**.

Giữ lại chỉ gây **đa cộng tuyến** mà không thêm thông tin.

In [5]:
DU_THUA = [
    ('orders',    'zip',            'trung 100% customers.zip qua customer_id'),
    ('payments',  'payment_method', 'trung 100% orders.payment_method qua order_id'),
    ('customers', 'city',           'suy tu geography.city qua zip'),
    ('reviews',   'customer_id',    'suy tu orders.customer_id qua order_id'),
    ('inventory', 'product_name',   'sao chep tu products'),
    ('inventory', 'category',       'sao chep tu products'),
    ('inventory', 'segment',        'sao chep tu products'),
]
for t, c, ly_do in DU_THUA:
    if c in b[t].columns:
        b[t] = b[t].drop(columns=[c])
        ghi(t, 'Loai cot du thua', c, ly_do)
        print('  ' + t + '.' + c)

print()
print('Da loai', len(DU_THUA), 'cot du thua')

  orders.zip
  payments.payment_method
  customers.city
  reviews.customer_id
  inventory.product_name
  inventory.category
  inventory.segment

Da loai 7 cot du thua


---

## Bước 5 — Loại cột suy diễn ⭐

**Cột suy diễn** tính lại được bằng công thức. Bước Từ điển đã xác nhận **5 công thức khớp 100%**.

| Cột | Công thức |
|---|---|
| `fill_rate` | `1 − stockout_days / 30` |
| `stockout_flag` | `stockout_days > 0` |
| `sell_through_rate` | `units_sold / (stock_on_hand + units_sold)` |
| `days_of_supply` | `round(stock_on_hand / (units_sold/30), 1)` |
| `payment_value` | `Σ(quantity × unit_price − discount_amount)` |

Cộng thêm `year` và `month` tách được từ `snapshot_date`.

In [6]:
SUY_DIEN = [
    ('inventory', 'fill_rate',         '= 1 - stockout_days/30'),
    ('inventory', 'stockout_flag',     '= (stockout_days > 0)'),
    ('inventory', 'sell_through_rate', '= units_sold/(stock_on_hand + units_sold)'),
    ('inventory', 'days_of_supply',    '= round(stock_on_hand/(units_sold/30), 1)'),
    ('inventory', 'year',              'tach tu snapshot_date'),
    ('inventory', 'month',             'tach tu snapshot_date'),
    ('payments',  'payment_value',     '= tong(quantity*unit_price - discount_amount)'),
]
for t, c, ct in SUY_DIEN:
    if c in b[t].columns:
        b[t] = b[t].drop(columns=[c])
        ghi(t, 'Loai cot suy dien', c, ct)
        print('  ' + t + '.' + c + '  ' + ct)

print()
print('Da loai', len(SUY_DIEN), 'cot suy dien')
print('Bang inventory con:', list(b['inventory'].columns))

  inventory.fill_rate  = 1 - stockout_days/30
  inventory.stockout_flag  = (stockout_days > 0)
  inventory.sell_through_rate  = units_sold/(stock_on_hand + units_sold)
  inventory.days_of_supply  = round(stock_on_hand/(units_sold/30), 1)
  inventory.year  tach tu snapshot_date
  inventory.month  tach tu snapshot_date
  payments.payment_value  = tong(quantity*unit_price - discount_amount)

Da loai 7 cot suy dien
Bang inventory con: ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'overstock_flag']


---

## Bước 6 — Xử lý giá trị thiếu **có ngữ nghĩa**

> Đây là chỗ dễ sai nhất. Không phải giá trị thiếu nào cũng là lỗi.

| Cột | Thiếu | Nghĩa thật | Cách xử lý |
|---|---:|---|---|
| `order_items.promo_id` | 61,3% | Dòng hàng **không có khuyến mại** | Thêm cờ nhị phân, **giữ nguyên** cột gốc |
| `order_items.promo_id_2` | 99,97% | Khuyến mại cộng dồn | **Loại bỏ** — chỉ 206 dòng có giá trị |
| `promotions.applicable_category` | 80% | **Áp dụng mọi danh mục** | Điền nhãn `ALL`, **tuyệt đối không xóa dòng** |

In [7]:
oi = b['order_items']
oi['co_khuyen_mai'] = oi.promo_id.notna().astype(int)
print('Them co co_khuyen_mai:', round(float(oi.co_khuyen_mai.mean()) * 100, 2), '% dong hang co KM')
ghi('order_items', 'Them co nhi phan', 'co_khuyen_mai (0/1)',
    'promo_id thieu 61,3% = "khong co khuyen mai", khong phai loi')

n_km2 = int(oi.promo_id_2.notna().sum())
oi = oi.drop(columns=['promo_id_2']); b['order_items'] = oi
print('Loai promo_id_2 — chi', n_km2, 'dong co gia tri')
ghi('order_items', 'Loai cot', 'promo_id_2', 'Rong 99,97%, chi ' + str(n_km2) + ' dong')

pr = b['promotions']
n_null = int(pr.applicable_category.isna().sum())
pr['applicable_category'] = pr.applicable_category.fillna('ALL')
print('Dien', n_null, 'o trong cua applicable_category thanh nhan ALL')
ghi('promotions', 'Dien nhan tuong minh', 'applicable_category: ' + str(n_null) + " o trong -> 'ALL'",
    'Thieu mang nghia "ap dung moi danh muc", KHONG duoc xoa dong')

Them co co_khuyen_mai: 38.66 % dong hang co KM
Loai promo_id_2 — chi 206 dong co gia tri
Dien 40 o trong cua applicable_category thanh nhan ALL


---

## Bước 7 — Thay `signup_date` bằng `first_order_date` ⭐

**Căn cứ:** bước Từ điển phát hiện **73,8% đơn hàng đặt TRƯỚC ngày đăng ký tài khoản** — bất khả thi
về nghiệp vụ. Cột này không dùng được cho bất kỳ đặc trưng nào.

**Không xóa hẳn** vì sẽ mất thông tin về khách hàng. Thay bằng **ngày đặt hàng đầu tiên**, suy trực tiếp
từ bảng `orders` — chỉ số này nhất quán về thời gian.

In [8]:
o, c = b['orders'], b['customers']

m = o[['customer_id','order_date']].merge(c[['customer_id','signup_date']], on='customer_id')
ty_le_sai = float((m.order_date < m.signup_date).mean())
print('Truoc khi sua: ', round(ty_le_sai * 100, 1), '% don dat TRUOC ngay dang ky')

lan_dau = o.groupby('customer_id')['order_date'].min().rename('first_order_date').reset_index()
c = c.merge(lan_dau, on='customer_id', how='left')
c['da_tung_mua'] = c.first_order_date.notna().astype(int)
c = c.drop(columns=['signup_date'])
b['customers'] = c

print('Sau khi sua:  0 % (theo dinh nghia, first_order_date <= moi don cua khach)')
print('Khach da tung mua:', format(int(c.da_tung_mua.sum()), ',d'), '/', format(len(c), ',d'))
ghi('customers', 'Thay the cot', 'signup_date -> first_order_date + da_tung_mua',
    str(round(ty_le_sai*100,1)) + '% don dat TRUOC ngay dang ky')
c.head(3)

Truoc khi sua:  73.8 % don dat TRUOC ngay dang ky
Sau khi sua:  0 % (theo dinh nghia, first_order_date <= moi don cua khach)
Khach da tung mua: 90,246 / 121,930


,customer_id,zip,gender,age_group,acquisition_channel,first_order_date,da_tung_mua
0,1,15201,Female,35-44,social_media,2012-07-25,1
1,2,15201,Female,45-54,email_campaign,2013-09-20,1
2,3,15201,Female,18-24,organic_search,2012-08-27,1


---

## Bước 8 — Đánh dấu SKU chết (không xóa)

**Căn cứ:** 814 trên 2.412 sản phẩm **chưa từng phát sinh giao dịch**, trong đó 654 sản phẩm có giá bất
thường dưới 100 — làm lệch mọi thống kê về giá và biên lợi nhuận.

**Vì sao không xóa:** chúng vẫn được bảng `inventory` tham chiếu. Xóa đi sẽ **tạo ra bản ghi mồ côi**.
Thay vào đó thêm cờ để lọc khi cần.

In [9]:
p = b['products']
da_ban = set(b['order_items'].product_id)
p['da_tung_ban'] = p.product_id.isin(da_ban).astype(int)
n_chet = int((p.da_tung_ban == 0).sum())
b['products'] = p

print('SKU chua tung ban:', n_chet, '/', len(p),
      '=', round(n_chet/len(p)*100, 1), '%')
print()
print('Gia trung vi theo nhom:')
print(p.groupby('da_tung_ban')['price'].median().round(2).to_string())
ghi('products', 'Them co nhi phan', 'da_tung_ban (0/1) — ' + str(n_chet) + ' SKU chet',
    'Khong xoa vi con duoc inventory tham chieu')

SKU chua tung ban: 814 / 2412 = 33.7 %

Gia trung vi theo nhom:
da_tung_ban
0      37.21
1    5505.13


---

# Kiểm chứng an toàn

> Làm sạch xong phải chứng minh **không phá vỡ gì**. Hai phép kiểm dưới đây là bắt buộc.

## Kiểm chứng 1 — Toàn vẹn tham chiếu vẫn nguyên

In [10]:
oi, o, p = b['order_items'], b['orders'], b['products']
pay, sh, r = b['payments'], b['shipments'], b['returns']
rv, inv, c, g, pr = b['reviews'], b['inventory'], b['customers'], b['geography'], b['promotions']

kt = [('order_items.order_id',   oi.order_id,          o.order_id),
      ('order_items.product_id', oi.product_id,        p.product_id),
      ('order_items.promo_id',   oi.promo_id.dropna(), pr.promo_id),
      ('orders.customer_id',     o.customer_id,        c.customer_id),
      ('customers.zip',          c.zip,                g.zip),
      ('payments.order_id',      pay.order_id,         o.order_id),
      ('shipments.order_id',     sh.order_id,          o.order_id),
      ('returns.order_id',       r.order_id,           o.order_id),
      ('returns.product_id',     r.product_id,         p.product_id),
      ('reviews.order_id',       rv.order_id,          o.order_id),
      ('reviews.product_id',     rv.product_id,        p.product_id),
      ('inventory.product_id',   inv.product_id,       p.product_id)]

kq = pd.DataFrame([{'Khoa ngoai': t, 'So ban ghi': len(con),
                    'Mo coi': int((~con.isin(set(cha))).sum())}
                   for t, con, cha in kt])
print('TONG:', format(kq['So ban ghi'].sum(), ',d'), 'ban ghi |',
      kq['Mo coi'].sum(), 'mo coi')
kq

TONG: 4,054,768 ban ghi | 0 mo coi


,Khoa ngoai,Mo coi,So ban ghi
0,order_items.order_id,0,714669
1,order_items.product_id,0,714669
2,order_items.promo_id,0,276316
3,orders.customer_id,0,646945
4,customers.zip,0,121930
5,payments.order_id,0,646945
6,shipments.order_id,0,566067
7,returns.order_id,0,39939
8,returns.product_id,0,39939
9,reviews.order_id,0,113551


### ✅ Vẫn **0 bản ghi mồ côi**

Việc làm sạch không phá vỡ quan hệ nào giữa các bảng.

## Kiểm chứng 2 — Biến mục tiêu vẫn tái tạo được

In [11]:
s = b['sales']
j = oi.merge(p[['product_id','cogs']], on='product_id') \
      .merge(o[['order_id','order_date']], on='order_id')
agg = j.assign(rev=j.quantity*j.unit_price, cog=j.quantity*j.cogs) \
       .groupby('order_date')[['rev','cog']].sum().round(2)
k = s.set_index('Date').join(agg)

print('Revenue: sai so tuyet doi lon nhat =', round(float((k.Revenue - k.rev).abs().max()), 4))
print('COGS   : sai so tuyet doi lon nhat =', round(float((k.COGS - k.cog).abs().max()), 4))
print('So ngay khop chinh xac:', int(((k.Revenue - k.rev).abs() < 0.02).sum()), '/', len(k))

Revenue: sai so tuyet doi lon nhat = 0.0
COGS   : sai so tuyet doi lon nhat = 0.0
So ngay khop chinh xac: 3833 / 3833


### ✅ Sai số vẫn bằng **0,00 trên cả 3.833 ngày**

Việc làm sạch không làm sai lệch biến mục tiêu.

---

# Báo cáo trước / sau

In [12]:
bc = pd.DataFrame([{
    'Bang': t,
    'Cot truoc': truoc[t][1], 'Cot sau': b[t].shape[1],
    'Chenh': b[t].shape[1] - truoc[t][1],
    'Dong truoc': truoc[t][0], 'Dong sau': b[t].shape[0],
} for t in TEN])

print('TONG COT :', bc['Cot truoc'].sum(), '->', bc['Cot sau'].sum(),
      '(' + str(bc['Chenh'].sum()) + ')')
print('TONG DONG:', format(bc['Dong truoc'].sum(), ',d'), '->',
      format(bc['Dong sau'].sum(), ',d'), '(khong mat dong nao)')
bc

TONG COT : 96 -> 84 (-12)
TONG DONG: 2,960,736 -> 2,960,736 (khong mat dong nao)


,Bang,Chenh,Cot sau,Cot truoc,Dong sau,Dong truoc
0,orders,-1,7,8,646945,646945
1,order_items,1,8,7,714669,714669
2,payments,-2,2,4,646945,646945
3,shipments,0,4,4,566067,566067
4,returns,0,7,7,39939,39939
5,reviews,-1,6,7,113551,113551
6,customers,0,7,7,121930,121930
7,products,1,9,8,2412,2412
8,geography,0,4,4,39948,39948
9,promotions,0,10,10,50,50


## Nhật ký thao tác

In [13]:
nk = pd.DataFrame(nhat_ky)
print('Tong so thao tac:', len(nk))
nk

Tong so thao tac: 22


,Bang,Can cu,Chi tiet,Thao tac
0,(nhieu bang),CSV luu ngay duoi dang chuoi,12 cot chuoi -> datetime,Ep kieu ngay
1,order_items,16 cap bi trung -> khong co khoa hop le,order_item_id (1..714669),Them khoa thay the
2,inventory,"Phuong sai = 0, khong mang thong tin",reorder_flag (luon = 0),Loai cot hang so
3,orders,trung 100% customers.zip qua customer_id,zip,Loai cot du thua
4,payments,trung 100% orders.payment_method qua order_id,payment_method,Loai cot du thua
5,customers,suy tu geography.city qua zip,city,Loai cot du thua
6,reviews,suy tu orders.customer_id qua order_id,customer_id,Loai cot du thua
7,inventory,sao chep tu products,product_name,Loai cot du thua
8,inventory,sao chep tu products,category,Loai cot du thua
9,inventory,sao chep tu products,segment,Loai cot du thua


---

# Ghi dữ liệu sạch

In [14]:
for t in TEN:
    b[t].to_csv(RA + t + '.csv', index=False)
nk.to_csv(RA + '_nhat_ky_lam_sach.csv', index=False)

print('Da ghi', len(TEN), 'tep vao', RA)
print('Nhat ky:', RA + '_nhat_ky_lam_sach.csv')

Da ghi 14 tep vao data_clean/
Nhat ky: data_clean/_nhat_ky_lam_sach.csv


---

# Tổng kết

| Chỉ tiêu | Trước | Sau |
|---|---:|---:|
| Số cột | 96 | **84** (−12) |
| Số dòng | 2.960.736 | **2.960.736** (không mất dòng nào) |
| Bản ghi mồ côi | 0 | **0** |
| Sai số tái tạo biến mục tiêu | 0,00 | **0,00** |

## Tám nhóm thao tác

| # | Thao tác | Căn cứ |
|---:|---|---|
| 1 | Ép kiểu 12 cột ngày | CSV lưu ngày dưới dạng chuỗi |
| 2 | Thêm khóa thay thế `order_item_id` | 16 cặp trùng → không có khóa hợp lệ |
| 3 | Loại 1 cột hằng số | `reorder_flag` luôn = 0 |
| 4 | Loại 7 cột dư thừa | Trùng khớp 100% với bảng khác |
| 5 | Loại 7 cột suy diễn | Công thức đã kiểm chứng khớp 100% |
| 6 | Xử lý 3 cột thiếu có ngữ nghĩa | Thiếu ≠ lỗi |
| 7 | Thay `signup_date` → `first_order_date` | 73,8% đơn đặt trước ngày đăng ký |
| 8 | Đánh dấu 814 SKU chết | Không xóa vì còn được tham chiếu |

## Ba quyết định đáng chú ý

**Không xóa dòng nào.** Dữ liệu bẩn ở đây là *cột thừa*, không phải *dòng lỗi* — nên xóa dòng là sai hướng.

**`signup_date` thay chứ không xóa.** Bỏ hẳn thì mất thông tin về khách hàng. Thay bằng `first_order_date`
tính từ chính bảng `orders` — nhất quán về thời gian, dùng được ngay cho đặc trưng thâm niên khách hàng.

**SKU chết đánh dấu chứ không xóa.** 814 sản phẩm này vẫn được `inventory` tham chiếu; xóa đi sẽ tạo bản
ghi mồ côi. Thay vào đó thêm cờ `da_tung_ban` để lọc khi tính thống kê giá.

---

**Bước tiếp theo:** xây dựng đặc trưng (feature engineering) trên bộ dữ liệu đã làm sạch.